# LLaVA-OneVision 7B Baseline on Eight Benchmark Types

This notebook evaluates the unmodified `llava-hf/llava-onevision-qwen2-7b-ov-hf`
checkpoint on the same eight benchmark types used by
`gemma4_31b_eight_types.ipynb`. Run it before fine-tuning to establish the
comparison baseline.

| Type | Task | Benchmark | Evaluation split |
|---|---|---|---|
| `L` | Labeling | `fashion_mnist` | `test` |
| `A` | Visual question answering | `docvqa` | `validation` |
| `B` | Bounding-box detection | `openimages_v4_detection` | `validation` |
| `C` | Captioning | `mscoco_caption` | `test` |
| `E` | Editing prompt reconstruction | `magicbrush` | `train` |
| `G` | Generation prompt reconstruction | `diffusiondb` | `train` |
| `P` | Preference | `pick_a_pic` | `train` |
| `R` | Rating | `tad66k` | `train` |


## 1. Environment

In [ ]:
from pathlib import Path
import getpass
import json
import os
import sys

candidate_roots = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next(
    (path for path in candidate_roots if (path / "models").is_dir() and (path / "benchmarks").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Run this notebook from this repository or one of its subdirectories.")
REPO_ROOT = REPO_ROOT.resolve()
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

requirements_path = REPO_ROOT / "requirements.txt"
fine_tuning_requirements = REPO_ROOT / "fine_tuning" / "requirements.txt"
%pip install -q -r "$requirements_path"
%pip install -q -r "$fine_tuning_requirements"

from huggingface_hub import login

HF_TOKEN = os.getenv("HF_TOKEN") or os.getenv("HUGGING_FACE_HUB_TOKEN")
if not HF_TOKEN:
    entered_token = getpass.getpass("HF token (press Enter for anonymous access): ").strip()
    if entered_token:
        HF_TOKEN = entered_token
        os.environ["HF_TOKEN"] = entered_token
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)

print("Repository:", REPO_ROOT)

## 2. Configuration

In [ ]:
MODEL_ID = "llava-hf/llava-onevision-qwen2-7b-ov-hf"
EVAL_SAMPLES_PER_TYPE = 100
USE_4BIT = True
OVERWRITE = False

BENCHMARK_SPECS = {
    "L": {"name": "fashion_mnist", "eval_split": "test"},
    "A": {"name": "docvqa", "eval_split": "validation"},
    "B": {"name": "openimages_v4_detection", "eval_split": "validation"},
    "C": {"name": "mscoco_caption", "eval_split": "test"},
    "E": {"name": "magicbrush", "eval_split": "train"},
    "G": {"name": "diffusiondb", "eval_split": "train"},
    "P": {"name": "pick_a_pic", "eval_split": "train"},
    "R": {"name": "tad66k", "eval_split": "train"},
}

OUTPUT_DIR = REPO_ROOT / "results" / "llava_onevision_7b_eight_types_baseline"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR

## 3. Benchmark registry

In [ ]:
from benchmarks import (
    DiffusionDBBenchmark,
    DocVQABenchmark,
    FashionMNISTBenchmark,
    MSCOCOCaptionBenchmark,
    MagicBrushBenchmark,
    OpenImagesV4DetectionBenchmark,
    PickAPicBenchmark,
    TAD66KBenchmark,
)

BENCHMARK_CLASSES = {
    "L": FashionMNISTBenchmark,
    "A": DocVQABenchmark,
    "B": OpenImagesV4DetectionBenchmark,
    "C": MSCOCOCaptionBenchmark,
    "E": MagicBrushBenchmark,
    "G": DiffusionDBBenchmark,
    "P": PickAPicBenchmark,
    "R": TAD66KBenchmark,
}

## 4. Load the base model

The default uses 4-bit inference so the 7B checkpoint fits on a wider range of GPUs.

In [ ]:
import torch
from transformers import AutoProcessor, BitsAndBytesConfig, LlavaOnevisionForConditionalGeneration

from models import LlavaOnevisionQwen2_7B

def load_benchmark_model(*, adapter_dir=None, name):
    model_kwargs = {
        "device_map": "auto",
        "low_cpu_mem_usage": True,
        "torch_dtype": torch.bfloat16,
        "attn_implementation": "sdpa",
    }
    if USE_4BIT:
        model_kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
        )

    processor_source = adapter_dir or MODEL_ID
    processor = AutoProcessor.from_pretrained(processor_source)
    model = LlavaOnevisionForConditionalGeneration.from_pretrained(MODEL_ID, **model_kwargs)
    if adapter_dir is not None:
        from peft import PeftModel
        model = PeftModel.from_pretrained(model, adapter_dir)
    model.eval()

    wrapper = LlavaOnevisionQwen2_7B.__new__(LlavaOnevisionQwen2_7B)
    wrapper.model_id = MODEL_ID
    wrapper.name = name
    wrapper.max_new_tokens = 64
    wrapper.stream = False
    wrapper.processor = processor
    wrapper.model = model
    return wrapper

In [ ]:
base_model = load_benchmark_model(name="llava-onevision-qwen2-7b-base")
base_model.name

## 5. Run the eight held-out evaluations

In [ ]:
import pandas as pd

from runners import BenchmarkRun, ModelRun, run_benchmark_matrix

def evaluate_eight_types(model, *, model_name, output_dir):
    output_dir.mkdir(parents=True, exist_ok=True)
    summaries = []
    for task_type, spec in BENCHMARK_SPECS.items():
        benchmark = BENCHMARK_CLASSES[task_type](
            split=spec["eval_split"],
            streaming=True,
        )
        model.max_new_tokens = benchmark.default_max_new_tokens
        result_path = output_dir / f"{model_name}_{benchmark.name}.json"
        if result_path.exists() and not OVERWRITE:
            summaries.append(
                {
                    "type": task_type,
                    "model": model_name,
                    "benchmark": benchmark.name,
                    "status": "skipped",
                    "results_path": str(result_path),
                }
            )
            continue
        try:
            summary = run_benchmark_matrix(
                models=[ModelRun(name=model_name, model=model)],
                benchmark_runs=[
                    BenchmarkRun(
                        benchmark=benchmark,
                        num_samples=EVAL_SAMPLES_PER_TYPE,
                    )
                ],
                output_dir=output_dir,
            )[0]
            summaries.append({"type": task_type, "status": "ok", **summary})
        except Exception as exc:
            summaries.append(
                {
                    "type": task_type,
                    "model": model_name,
                    "benchmark": benchmark.name,
                    "status": "error",
                    "error": f"{type(exc).__name__}: {exc}",
                }
            )
        (output_dir / "run_summary.json").write_text(
            json.dumps(summaries, indent=2),
            encoding="utf-8",
        )
        print(task_type, benchmark.name, summaries[-1]["status"], flush=True)
    return pd.DataFrame(summaries)

In [ ]:
baseline_summary = evaluate_eight_types(
    base_model,
    model_name="llava-onevision-qwen2-7b-base",
    output_dir=OUTPUT_DIR,
)
baseline_summary

## Notes

- Keep `EVAL_SAMPLES_PER_TYPE`, `USE_4BIT`, and the split map unchanged when
  comparing with the fine-tuned notebook.
- `magicbrush`, `diffusiondb`, `pick_a_pic`, and `tad66k` expose their canonical
  evaluation through `train`; the fine-tuning notebook reserves an initial slice
  before exporting training data, but dataset adapters with count-dependent
  sampling should still be treated as a possible overlap risk.
- Some datasets require Hugging Face authentication or substantial downloads.